In [7]:
import pandas as pd 
import numpy as np
import ast

import os

movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")
# print(movies.head(3))

movies.describe()

## merge the dataset together
data = movies.merge(credits, on='title')
# data.head(3)
new_data = data[['genres','id','keywords','overview','title','release_date', 'cast','crew']]

# new_data.shape
new_data.isnull().sum() ## check null values in the dataset

## rremove the null value from the dataset
new_data.dropna(inplace=True)
new_data.reset_index(drop=True, inplace=True)
new_data.isnull().sum() ## check again null values removed or not with the help of isnull()


## check any duplicate value in the dataset
new_data.duplicated().sum()


## Preprocessing in Recommandation System
new_data['genres'][0]


## Extract the data for genres
def extract_genres(col):
    l= []
    for i in ast.literal_eval(col):
        l.append(i['name'])
    return l

new_data['genres'] = new_data['genres'].apply(extract_genres)  
new_data['keywords'] = new_data['keywords'].apply(extract_genres)
# new_data['cast'] = new_data[]


new_data['release_date'] = pd.to_datetime(new_data['release_date']) 

release_year = new_data['release_date'].dt.year
new_data['cast'][1]


## get cast name for the data preprocessing

def get_cast_name(col):
    l = []

    for i in ast.literal_eval(col)[:3]:
        l.append(i['name'])

    return l

new_data['cast'] = new_data['cast'].apply(get_cast_name)  

## get crew data for the data preprocessing

def get_crew_data(col):
    l = []
    for i in ast.literal_eval(col):
        if i['job'] == 'Director':
            l.append(i['name'])
    return l


new_data['crew'] = new_data['crew'].apply(get_crew_data)
# new_data.head(3)


## Convert overview string data into a list 
new_data['overview'] = new_data['overview'].apply(lambda x: x.split())

## remove spaces from the list

new_data['cast'] = new_data['cast'].apply(
    lambda cast_list: [cast.replace(" ", "") for cast in cast_list]
)
new_data['crew'] = new_data['crew'].apply(
    lambda crew_list: [crew.replace(" ", "") for crew in crew_list]
)
new_data['genres'] = new_data['genres'].apply(
    lambda genres_list: [genres.replace(" ", "") for genres in genres_list]
)
new_data['keywords'] = new_data['keywords'].apply(
    lambda keywords_list: [keywords.replace(" ", "") for keywords in keywords_list]
)

new_data['overview'] = new_data['overview'].apply(
    lambda overview_list: [overview.replace(" ", "") for overview in overview_list]
)


new_data.head(3)

## concat the data
new_data['tags'] = new_data['genres'] + new_data['keywords'] + new_data['cast'] + new_data['crew'] + new_data['overview']
new_data.drop(columns=['genres','keywords','cast','crew','overview','release_date'],inplace=True)

new_data['tags'] = new_data['tags'].apply(lambda x: " " .join(x))

new_data['tags'] = new_data['tags'].apply(lambda x:x.lower())
new_data['title'] = new_data['title'].apply(lambda x:x.lower())
new_data['tags'][0]
 








'action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver jamescameron in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.'

NLP Preprocessing and Feature Engineering
Text Vectorization 

Bag of Words (BoW) Technique

Bag of Words is a technique used in Natural Language Processing (NLP) to convert text into numbers, so a machine-learning model can work with text.
Movie 1: action adventure superhero
Movie 2: action superhero
Movie 3: comedy romance

First, BoW creates a vocabulary containing all unique words:
action
adventure
superhero
comedy
romance
Then it represents each movie using the frequency of each word:

Movie	action	adventure	superhero	comedy	romance
Movie 1 	1	   1	        1	       0	   0
Movie 2	    1	   0	        1	       0	   0
Movie 3   	0	   0	        0	       1	   1

Because BoW doesn't care about the order of words.

"action adventure movie"  &&  "movie adventure action"


Stop Word Removal
Stemming

In [8]:
## Stop Word Removal

!pip install nltk


In [ ]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000,stop_words='english') 

from sklearn.metrics.pairwise import cosine_similarity


## Pipeline will be for Recommandation Project
## Movie Text  ---> Clean Text ---> Lowercase ---> Remove Unnecessary spaces/symbol ---> stemming ---> Bag of Words ---> Numerical Vectors --->Cosine similiarity   ---> Recommmandation


def steamming(text):
    L = []
    for i in text.split():
        L.append(ps.stem(i))
    return " ".join(L)    
print(new_data['tags'].isnull().sum())


new_data['tags'] = new_data['tags'].apply(steamming) 
# new_data['tags']


## Vectorization : 

vectors = cv.fit_transform(new_data['tags'])
vectors = vectors.toarray()
vectors = cv.fit_transform(new_data['tags'])


similiarties = cosine_similarity(vectors)
# similiarties[0]

def recommand(movie):
    movie_index = new_data[
        new_data['title'].str.lower() == movie.lower()
    ].index[0]

    distances = similiarties[movie_index]

    recommandations = sorted(
        enumerate(distances),
        reverse=True,
        key=lambda x: x[1]
    )

    for i in recommandations[1:6]:
        print(new_data.iloc[i[0]].title)


recommand('Avatar')





0


aliens vs predator: requiem
aliens
falcon rising
independence day
titan a.e.


,id,title,tags
0,19995,avatar,action adventur fantasi sciencefict culturecla...
1,285,pirates of the caribbean: at world's end,adventur fantasi action ocean drugabus exotici...
2,206647,spectre,action adventur crime spi basedonnovel secreta...
3,49026,the dark knight rises,action crime drama thriller dccomic crimefight...
4,49529,john carter,action adventur sciencefict basedonnovel mar m...
...,...,...,...
4800,9367,el mariachi,action crime thriller unitedstates–mexicobarri...
4801,72766,newlyweds,comedi romanc edwardburn kerrybishé marshadiet...
4802,231617,"signed, sealed, delivered",comedi drama romanc tvmovi date loveatfirstsig...
4803,126186,shanghai calling,danielhenney elizacoup billpaxton danielhsia w...


In [11]:
import pickle

with open('movies.pickle', 'wb') as movies:
    pickle.dump(new_data, movies)

In [12]:
import joblib 
joblib.dump(similiarties, "simlarty.joblib", compress=3)

['simlarty.joblib']